In [1]:
import os
import random
import numpy as no

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data

import torchvision.transforms as transforms
import torchvision.datasets as datasets

from torchsummary import summary

import matplotlib.pyplot as plt
from PIL import Image

In [2]:
ROOT = 'data'
train_data = datasets.MNIST(root=ROOT, train=True, download=True)
test_data = datasets.MNIST(root=ROOT, train=False, download=True)

In [3]:
VALID_RATIO = 0.9

n_train_examples = int(len(train_data) * VALID_RATIO)
n_valid_examples = len(train_data) - n_train_examples

train_data, valid_data = data.random_split(train_data, [n_train_examples, n_valid_examples])

# compute mean and std of the training data
mean = train_data.dataset.data.float().mean() / 255
std = train_data.dataset.data.float().std() / 255

train_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean], std=[std])
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean], std=[std])
])

train_data.dataset.transform = train_transforms
valid_data.dataset.transform = test_transforms

# Create dataloaders

BATCH_SIZE = 256

train_dataloader = data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_dataloader = data.DataLoader(valid_data, batch_size=BATCH_SIZE)

In [35]:
# Xây dựng mô hình LeNet

class LeNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels=1, out_channels=6, kernel_size=5, padding = 'same'
        )
        self.avgpool1 = nn.AvgPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size = 5)
        self.avgpool2 = nn.AvgPool2d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.fc_1 = nn.Linear(in_features=16*5*5, out_features=120)
        self.fc_2 = nn.Linear(in_features=120, out_features=84)
        self.fc_3 = nn.Linear(in_features=84, out_features=10)
        
    def forward(self, inputs):
        outputs = self.conv1(inputs)
        outputs = self.avgpool1(outputs)
        outputs = F.relu(outputs)
        outputs = self.conv2(outputs)
        outputs = self.avgpool2(outputs)
        outputs = F.relu(outputs)
        outputs = self.flatten(outputs)
        outputs = self.fc_1(outputs)
        outputs = F.relu(outputs)
        outputs = self.fc_2(outputs)
        outputs = F.relu(outputs)
        outputs = self.fc_3(outputs)
        return outputs
        

In [36]:
num_classes = len(train_data.dataset.classes)
num_classes

10

In [37]:
model = LeNet(num_classes)
model

LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=same)
  (avgpool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (avgpool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc_1): Linear(in_features=400, out_features=120, bias=True)
  (fc_2): Linear(in_features=120, out_features=84, bias=True)
  (fc_3): Linear(in_features=84, out_features=10, bias=True)
)

In [38]:
inputs, labels = next(iter(train_dataloader))

In [39]:
prediction = model(inputs)
prediction.shape

torch.Size([256, 10])

In [5]:
import time
def train(model, optimizer, criterion, train_dataloader, device, epoch = 0, log_interval = 50):
    model.train()
    total_acc, total_count = 0, 0
    losses = []
    start_time = time.time()
    
    for idx, (inputs, labels) in enumerate(train_dataloader):
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        predictions = model(inputs)
        
        # compute loss
        loss = criterion(predictions, labels)
        losses.append(loss.item())
        
        # backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        total_acc += (predictions.argmax(1) == labels).sum().item()
        total_count += labels.size(0)
        
        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time()
            print(
                "| epoch {:3d} | {:5d}/{:5d} batches "
                "| accuracy {:8.3}".format(
                    epoch, idx, len(train_dataloader), total_acc / total_count
                )
            )
            total_acc, total_count = 0, 0
            start_time = time.time()
    epoch_acc = total_acc / total_count
    epoch_loss = sum(losses) / len(losses)
    return epoch_acc, epoch_loss

In [40]:
# Training
num_classes = len(train_data.dataset.classes)
device = torch.device('cuda')
lenet_model = LeNet(num_classes).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(lenet_model.parameters())

num_epochs = 50
save_model = './model'

train_accs, train_losses = [], []
eval_accs, eval_losses = [], []
best_loss_eval = 100

In [44]:
criterion(prediction, labels)

tensor(2.3100, grad_fn=<NllLossBackward0>)

In [46]:
train_acc, train_loss = train(lenet_model, optimizer, criterion, train_dataloader, device, epoch = 0, log_interval = 50)

| epoch   0 |    50/  211 batches | accuracy    0.943
| epoch   0 |   100/  211 batches | accuracy    0.953
| epoch   0 |   150/  211 batches | accuracy    0.958
| epoch   0 |   200/  211 batches | accuracy    0.958


In [49]:
def evaluate (model, criterion, valid_dataloader, device = 'cuda'):
    model.eval()
    total_acc, total_count = 0,0
    losses = []

    with torch.no_grad():
        for idx, (inputs, labels) in enumerate (valid_dataloader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            # predictions
            predictions = model(inputs)

            # compute loss
            loss = criterion(predictions, labels)
            losses.append(loss.item())

            total_acc += (predictions.argmax(1) == labels).sum().item()
            total_count += labels.size (0)

    epoch_acc = total_acc / total_count
    epoch_loss = sum (losses) / len(losses)
    return epoch_acc, epoch_loss

In [51]:
eval_acc, eval_loss = evaluate(lenet_model, criterion, valid_dataloader, device = 'cuda')
eval_acc, eval_loss

(0.9601666666666666, 0.13983569449434677)

In [53]:
num_epochs = 10
import os
if not os.path.exists('./model'):
    os.makedirs('./model')
save_model = './model'

train_accs, train_losses = [], []
eval_accs, eval_losses = [], []
best_loss_eval = 100

for epoch in range(1, num_epochs + 1):
    epoch_start_time = time.time()

    # Training
    train_acc, train_loss = train(lenet_model, optimizer, criterion, train_dataloader, device, epoch)
    train_accs.append(train_acc)
    train_losses.append(train_loss)

    # Evaluation
    eval_acc, eval_loss = evaluate(lenet_model, criterion, valid_dataloader)
    eval_accs.append(eval_acc)
    eval_losses.append(eval_loss)

    # Save best model
    if eval_loss < best_loss_eval:
        torch.save(lenet_model.state_dict(), save_model + '/lenet_model.pt')

    # Print loss, acc end epoch
    print("-" * 59)
    print(
        "| End of epoch {:3d} | Time: {:5.2f}s | Train Accuracy {:8.3f} | Train Loss {:8.3f} \n"
        "| Valid Accuracy {:8.3f} | Valid Loss {:8.3f} ".format(
            epoch, time.time() - epoch_start_time, train_acc, train_loss, eval_acc, eval_loss
        )
    )
    print("-" * 59)

| epoch   1 |    50/  211 batches | accuracy    0.974
| epoch   1 |   100/  211 batches | accuracy    0.976
| epoch   1 |   150/  211 batches | accuracy    0.975
| epoch   1 |   200/  211 batches | accuracy    0.976
-----------------------------------------------------------
| End of epoch   1 | Time: 14.95s | Train Accuracy    0.981 | Train Loss    0.078 
| Valid Accuracy    0.980 | Valid Loss    0.067 
-----------------------------------------------------------
| epoch   2 |    50/  211 batches | accuracy    0.979
| epoch   2 |   100/  211 batches | accuracy     0.98
| epoch   2 |   150/  211 batches | accuracy    0.979
| epoch   2 |   200/  211 batches | accuracy    0.982
-----------------------------------------------------------
| End of epoch   2 | Time: 14.09s | Train Accuracy    0.981 | Train Loss    0.063 
| Valid Accuracy    0.985 | Valid Loss    0.056 
-----------------------------------------------------------
| epoch   3 |    50/  211 batches | accuracy    0.983
| epoch   

In [55]:
# Load best model (nếu bạn đã lưu)
lenet_model.load_state_dict(torch.load(save_model + '/lenet_model.pt'))
lenet_model.eval()

# Đánh giá mô hình trên tập test
test_acc, test_loss = evaluate(lenet_model, criterion, valid_dataloader)

print(f"Độ chính xác trên tập test: {test_acc:.4f}")
print(f"Loss trên tập test: {test_loss:.4f}")

C:\Users\HP\AppData\Local\Temp\ipykernel_22584\1147517825.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  lenet_model.load_state_dict(torch.load(save_model + '/lenet_mod

Độ chính xác trên tập test: 0.9892
Loss trên tập test: 0.0363
